<a href="https://colab.research.google.com/github/kunjana30/Internship-Project-2/blob/main/Task5_Content_Rating_Genre_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 5 (Advanced) – Content Rating & Genre Analysis

**Goal:** Examine content ratings and genres to understand **audience targeting**.

**Prerequisite:** run the Task 1 notebook first (it creates `netflix_cleaned.csv`).

| Step | What we do |
|------|-----------|
| 1 | Analyse rating categories |
| 2 | Identify the most common genres (and genre combinations) |
| 3 | Compare ratings across content types |
| 4 | Create **interactive** visualisations (Plotly) |
| 5 | Present audience insights |

> 📦 **This notebook uses Plotly** for interactive charts (hover, zoom, click legend items to hide/show). Google Colab already has it installed. If you see
> `ModuleNotFoundError: No module named 'plotly'`, run `pip install plotly` in your terminal (or `%pip install plotly` in a cell), then restart the kernel.
>
> 💾 Every chart is also saved as a standalone `.html` file in `outputs/task5_rating_genre/` – handy for screenshots, and it opens in any browser even where notebooks can't show interactive charts (e.g. GitHub).

### ☁️ Google Colab / Drive setup – run this cell first
Colab's storage is **temporary**: files in `/content` disappear when the session ends, and each notebook runs in its own session.
So this cell connects to your **Google Drive** and works inside a folder called `Netflix_Analysis`, keeping `Dataset.csv`,
`netflix_cleaned.csv` (created by Task 1) and all outputs in one permanent place.

* A pop-up will ask you to allow Drive access - choose your account and click **Allow**.
* Make sure `MyDrive/Netflix_Analysis/` contains `Dataset.csv` (and `netflix_cleaned.csv` once Task 1 has been run).
* **Running locally in Jupyter?** This cell does nothing outside Colab, so you can leave it as it is.
* **No Drive?** Set `USE_DRIVE = False` and upload the files with the Files panel (folder icon on the left). They are deleted when the runtime resets.

In [ ]:
import os, sys

USE_DRIVE = True                                          # set to False to skip Google Drive
PROJECT_DIR = "/content/drive/MyDrive/Netflix_Analysis"   # the folder in your Drive that holds the project files

if "google.colab" in sys.modules and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files here    :", sorted(os.listdir(".")))

Mounted at /content/drive
Working folder: /content/drive/MyDrive/Netflix_Analysis
Files here    : ['Dataset.csv', 'Task1_Data_Cleaning_Preparation.ipynb', 'Task2_Content_Type_Analysis.ipynb', 'Task3_Country_Wise_Analysis.ipynb', 'Task4_Trend_Analysis_Release_Year.ipynb', 'Task5_Content_Rating_Genre_Analysis.ipynb', 'Task6_Business_Insights_Report.ipynb', 'netflix_cleaned.csv', 'outputs']


---
## Step 0 – Setup and load the cleaned dataset

In [ ]:
import warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

NETFLIX_RED, NETFLIX_DARK = "#E50914", "#221F1F"
TYPE_COLORS = {"Movie": NETFLIX_RED, "TV Show": NETFLIX_DARK}
GROUP_ORDER = ["Kids", "Older Kids", "Teens", "Adults", "Not Rated"]
GROUP_COLORS = {"Kids": "#2E86AB", "Older Kids": "#7FB7BE", "Teens": "#F6AE2D", "Adults": NETFLIX_RED, "Not Rated": "#B3B3B3"}
RATING_ORDER = ["TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G", "G", "TV-PG", "PG", "TV-14", "PG-13", "TV-MA", "R", "NC-17", "NR"]   # roughly youngest -> oldest

CLEAN_PATH = Path("netflix_cleaned.csv")                # created by Task 1
OUT_DIR = Path("outputs/task5_rating_genre")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not CLEAN_PATH.exists():
    raise FileNotFoundError("'netflix_cleaned.csv' not found. Run the Task 1 notebook first and keep this notebook in the same folder.")

def show(fig, name):
    """Apply a consistent look, save a standalone HTML copy, then display the interactive figure."""
    fig.update_layout(template="plotly_white", font=dict(size=13), title_font=dict(size=18), margin=dict(t=70, l=20, r=20, b=20))
    fig.write_html(OUT_DIR / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

In [ ]:
df = pd.read_csv(CLEAN_PATH, parse_dates=["date_added"])
print(f"Loaded {len(df):,} titles x {df.shape[1]} columns")
df.head()

Loaded 8,786 titles x 18 columns


,show_id,type,title,director,country,date_added,year_added,month_added,month_name,release_year,rating,rating_group,duration,duration_min,seasons,listed_in,n_genres,primary_genre
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2021,9,September,2020,PG-13,Teens,90 min,90.0,NaN,Documentaries,1,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,9,September,2021,TV-MA,Adults,1 Season,NaN,1.0,"Crime TV Shows, International TV Shows, TV Act...",3,Crime TV Shows
2,s11,TV Show,"Vendetta: Truth, Lies and The Mafia",Unknown,Pakistan,2021-09-24,2021,9,September,2021,TV-MA,Adults,1 Season,NaN,1.0,"Crime TV Shows, Docuseries, International TV S...",3,Crime TV Shows
3,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha",Unknown,2021-09-24,2021,9,September,2021,PG,Older Kids,91 min,91.0,NaN,Children & Family Movies,1,Children & Family Movies
4,s5,TV Show,Kota Factory,Unknown,India,2021-09-24,2021,9,September,2021,TV-MA,Adults,2 Seasons,NaN,2.0,"International TV Shows, Romantic TV Shows, TV ...",3,International TV Shows


---
## Step 1 – Analyse rating categories
A rating tells us **who a title is meant for**. In Task 1 we grouped the 13 rating codes into 5 audience groups:

| Audience group | Ratings |
|---|---|
| Kids | TV-Y, TV-G, G |
| Older Kids | TV-Y7, TV-Y7-FV, TV-PG, PG |
| Teens | PG-13, TV-14 |
| Adults | R, TV-MA, NC-17 |
| Not Rated | NR |

In [ ]:
rating_counts = df["rating"].value_counts().rename_axis("rating").reset_index(name="titles")
rating_counts["share_%"] = (rating_counts["titles"] / len(df) * 100).round(1)
rating_counts = rating_counts.merge(df[["rating", "rating_group"]].drop_duplicates(), on="rating")

group_counts = df["rating_group"].value_counts().reindex(GROUP_ORDER).rename_axis("audience_group").reset_index(name="titles")
group_counts["share_%"] = (group_counts["titles"] / len(df) * 100).round(1)

display(rating_counts)
group_counts

,rating,titles,share_%,rating_group
0,TV-MA,3204,36.5,Adults
1,TV-14,2155,24.5,Teens
2,TV-PG,861,9.8,Older Kids
3,R,798,9.1,Adults
4,PG-13,490,5.6,Teens
5,TV-Y7,333,3.8,Older Kids
6,TV-Y,306,3.5,Kids
7,PG,287,3.3,Older Kids
8,TV-G,220,2.5,Kids
9,NR,82,0.9,Not Rated


,audience_group,titles,share_%
0,Kids,567,6.5
1,Older Kids,1487,16.9
2,Teens,2645,30.1
3,Adults,4005,45.6
4,Not Rated,82,0.9


In [ ]:
fig = px.bar(
    rating_counts, x="rating", y="titles", color="rating_group", text_auto=True,
    color_discrete_map=GROUP_COLORS, hover_data=["share_%"],
    category_orders={"rating": rating_counts["rating"].tolist(), "rating_group": GROUP_ORDER},
    title="Number of titles by rating (coloured by audience group)",
)
fig.update_layout(xaxis_title="Rating", yaxis_title="Number of titles", legend_title="Audience group")
show(fig, "01_ratings")

---
## Step 2 – Identify the most common genres
The `listed_in` column holds **several genres per title** (`"Dramas, International Movies"`). To count genres we must **split → explode**
so each genre gets its own row. A title with 3 genres is therefore counted 3 times, so genre counts add up to more than the number of titles.

> Note: some Netflix "genres" such as *International Movies* are really **catalogue categories** (where a film is from) rather than story genres. We keep them, but read them with that in mind.

In [ ]:
genres = (df.assign(genre=df["listed_in"].str.split(","))
            .explode("genre")
            .reset_index(drop=True))                       # always reset the index after explode
genres["genre"] = genres["genre"].str.strip()

print(f"{len(df):,} titles -> {len(genres):,} title-genre rows ({genres['genre'].nunique()} distinct genres, {df['n_genres'].mean():.2f} genres per title on average)")

genre_counts = genres["genre"].value_counts().rename_axis("genre").reset_index(name="titles")
genre_counts["share_of_titles_%"] = (genre_counts["titles"] / len(df) * 100).round(1)
genre_counts.head(15)

8,786 titles -> 19,284 title-genre rows (42 distinct genres, 2.19 genres per title on average)


,genre,titles,share_of_titles_%
0,International Movies,2751,31.3
1,Dramas,2423,27.6
2,Comedies,1673,19.0
3,International TV Shows,1348,15.3
4,Documentaries,869,9.9
5,Action & Adventure,859,9.8
6,TV Dramas,761,8.7
7,Independent Movies,755,8.6
8,Children & Family Movies,641,7.3
9,Romantic Movies,616,7.0


In [ ]:
fig = px.bar(genre_counts.head(15), x="titles", y="genre", orientation="h", text_auto=True,
             hover_data=["share_of_titles_%"], title="Top 15 genres on Netflix (movies + TV shows)")
fig.update_traces(marker_color=NETFLIX_RED)
fig.update_layout(yaxis=dict(categoryorder="total ascending", title=""), xaxis_title="Number of titles")
show(fig, "02_top_genres")

Movies and TV shows use **different genre labels** (`Dramas` vs `TV Dramas`), so we look at the top genres *within* each type:

In [ ]:
genre_type = genres.groupby(["type", "genre"]).size().reset_index(name="titles")
top_gt = genre_type.sort_values(["type", "titles"], ascending=[True, False]).groupby("type").head(10)

fig = px.bar(top_gt, x="titles", y="genre", color="type", facet_col="type", orientation="h", text_auto=True,
             color_discrete_map=TYPE_COLORS, title="Top 10 genres within each content type")
fig.update_yaxes(matches=None, showticklabels=True, categoryorder="total ascending", title="")
fig.update_xaxes(title="Number of titles")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))     # "type=Movie" -> "Movie"
fig.update_layout(showlegend=False)
show(fig, "03_top_genres_by_type")

### Which genres appear together?
Counting genre **pairs** reveals how titles are positioned (e.g. `Dramas + International Movies`).

In [ ]:
pair_counts = Counter()
for genre_list in df["listed_in"].str.split(","):
    names = sorted({g.strip() for g in genre_list})
    pair_counts.update(combinations(names, 2))              # every unordered pair of genres inside one title

pairs = pd.DataFrame([{"genre_pair": f"{a}  +  {b}", "titles": n} for (a, b), n in pair_counts.most_common(12)])

fig = px.bar(pairs, x="titles", y="genre_pair", orientation="h", text_auto=True, title="12 most common genre combinations")
fig.update_traces(marker_color=NETFLIX_DARK)
fig.update_layout(yaxis=dict(categoryorder="total ascending", title=""), xaxis_title="Number of titles")
show(fig, "04_genre_pairs")

---
## Step 3 – Compare ratings across content types
Because there are more movies than TV shows, we compare **percentages within each type** rather than raw counts.

In [ ]:
rating_by_type_pct = (pd.crosstab(df["rating"], df["type"], normalize="columns") * 100).reindex(RATING_ORDER).round(1)
group_by_type_pct = (pd.crosstab(df["rating_group"], df["type"], normalize="columns") * 100).reindex(GROUP_ORDER).round(1)

display(rating_by_type_pct)
group_by_type_pct

type,Movie,TV Show
rating,,
TV-Y,2.1,6.6
TV-Y7,2.3,7.3
TV-Y7-FV,0.1,0.0
TV-G,2.1,3.5
G,0.7,0.0
TV-PG,8.8,12.1
PG,4.7,0.0
TV-14,23.3,27.4
PG-13,8.0,0.0


type,Movie,TV Show
rating_group,,
Kids,4.9,10.1
Older Kids,15.9,19.4
Teens,31.3,27.4
Adults,46.7,43.0
Not Rated,1.3,0.2


In [ ]:
rating_long = rating_by_type_pct.reset_index().melt(id_vars="rating", var_name="type", value_name="pct_of_type")

fig = px.bar(rating_long, x="rating", y="pct_of_type", color="type", barmode="group", text_auto=".0f",
             color_discrete_map=TYPE_COLORS, category_orders={"rating": RATING_ORDER},
             title="Rating mix: Movies vs TV Shows (% of each type's titles)")
fig.update_layout(xaxis_title="Rating (youngest -> oldest)", yaxis_title="% of titles within type", legend_title="")
show(fig, "05_ratings_by_type")

---
## Step 4 – Create interactive visualisations
**Try it:** hover over shapes for exact numbers, **click** a sunburst/treemap segment to zoom in, and click legend entries to hide/show a group.

### 4a. Sunburst – type → audience group → rating

In [ ]:
sun = df.groupby(["type", "rating_group", "rating"]).size().reset_index(name="titles")

fig = px.sunburst(sun, path=["type", "rating_group", "rating"], values="titles", color="type",
                  color_discrete_map=TYPE_COLORS, title="Who is Netflix content for? (click a segment to zoom)")
fig.update_traces(textinfo="label+percent parent")
fig.update_layout(height=650)
show(fig, "06_sunburst_audience")

### 4b. Treemap – genres sized by number of titles

In [ ]:
fig = px.treemap(genre_type, path=["type", "genre"], values="titles", color="type",
                 color_discrete_map=TYPE_COLORS, title="Genre landscape (box size = number of titles)")
fig.update_traces(textinfo="label+value")
fig.update_layout(height=650)
show(fig, "07_treemap_genres")

### 4c. Heatmap – which audience does each genre target?
Each **row adds up to 100%**: e.g. what share of *Children & Family Movies* titles is rated Kids vs Teens vs Adults.

In [ ]:
top_genres = genre_counts.head(15)["genre"].tolist()
gm = genres[genres["genre"].isin(top_genres)]

genre_audience = (pd.crosstab(gm["genre"], gm["rating_group"], normalize="index") * 100)
genre_audience = genre_audience.reindex(columns=GROUP_ORDER, fill_value=0).loc[top_genres].round(0)

fig = px.imshow(genre_audience, text_auto=".0f", aspect="auto", color_continuous_scale="Reds",
                labels=dict(x="Audience group", y="Genre", color="% of genre"),
                title="Audience mix of the 15 most common genres (% of each genre's titles)")
fig.update_layout(height=620)
show(fig, "08_genre_audience_heatmap")

### 4d. How has the audience mix changed over time?
A 100% stacked area chart of audience groups by **release year** (complete years only; click the legend to isolate a group).

In [ ]:
LAST_COMPLETE = int(df["date_added"].max().year) - 1        # the dataset stops in Sept of the final year

mix = (df[df["release_year"].between(2008, LAST_COMPLETE)]
         .groupby(["release_year", "rating_group"]).size().reset_index(name="titles"))

fig = px.area(mix, x="release_year", y="titles", color="rating_group", groupnorm="percent",
              color_discrete_map=GROUP_COLORS, category_orders={"rating_group": GROUP_ORDER},
              title=f"Audience mix of titles by release year (2008-{LAST_COMPLETE})")
fig.update_layout(xaxis_title="Release year", yaxis_title="% of titles released", legend_title="Audience group")
show(fig, "09_audience_mix_over_time")

---
## Step 5 – Present audience insights
### Audience segmentation table

In [ ]:
top_genre_by_group = genres.groupby("rating_group")["genre"].agg(lambda s: ", ".join(s.value_counts().head(3).index))

segments = df.groupby("rating_group").agg(
    titles=("show_id", "count"),
    movie_share_pct=("type", lambda s: (s == "Movie").mean() * 100),
    median_release_year=("release_year", "median"),
).reindex(GROUP_ORDER)
segments["share_of_catalogue_%"] = segments["titles"] / len(df) * 100
segments["top_3_genres"] = top_genre_by_group
segments = segments.round({"movie_share_pct": 0, "share_of_catalogue_%": 1})

# save tables for your report
segments.to_csv(OUT_DIR / "audience_segments.csv")
genre_counts.to_csv(OUT_DIR / "genre_counts.csv", index=False)
rating_by_type_pct.to_csv(OUT_DIR / "rating_by_type_pct.csv")
genre_audience.to_csv(OUT_DIR / "genre_audience_heatmap.csv")
segments

,titles,movie_share_pct,median_release_year,share_of_catalogue_%,top_3_genres
rating_group,,,,,
Kids,567,53.0,2018.0,6.5,"Kids' TV, Children & Family Movies, Comedies"
Older Kids,1487,65.0,2016.0,16.9,"Children & Family Movies, Comedies, Internatio..."
Teens,2645,72.0,2016.0,30.1,"International Movies, Dramas, Comedies"
Adults,4005,71.0,2017.0,45.6,"International Movies, Dramas, International TV..."
Not Rated,82,95.0,2014.5,0.9,"International Movies, Dramas, Documentaries"


In [ ]:
mix_pct = mix.pivot(index="release_year", columns="rating_group", values="titles").fillna(0)
mix_pct = mix_pct.div(mix_pct.sum(axis=1), axis=0) * 100
start_year = 2010

top_rating = rating_counts.iloc[0]
adult_movie, adult_tv = group_by_type_pct.loc["Adults", "Movie"], group_by_type_pct.loc["Adults", "TV Show"]
kids_movie = group_by_type_pct.loc[["Kids", "Older Kids"], "Movie"].sum()
kids_tv = group_by_type_pct.loc[["Kids", "Older Kids"], "TV Show"].sum()
top_pair = pairs.iloc[0]

insights = [
    f"**Mature content dominates.** {top_rating['rating']} is the most common rating ({top_rating['share_%']:.0f}% of titles), and {group_counts.set_index('audience_group').loc['Adults', 'share_%']:.0f}% of the catalogue is Adult-rated.",
    f"**TV shows skew younger than movies.** {kids_tv:.0f}% of TV titles target Kids/Older Kids versus {kids_movie:.0f}% of movies, while Adult-rated content is {adult_movie:.0f}% of movies and {adult_tv:.0f}% of TV.",
    f"**The audience mix has shifted towards adults.** Adult-rated titles were {mix_pct.loc[start_year, 'Adults']:.0f}% of {start_year} releases and {mix_pct.loc[LAST_COMPLETE, 'Adults']:.0f}% of {LAST_COMPLETE} releases, while Teen-rated titles fell from {mix_pct.loc[start_year, 'Teens']:.0f}% to {mix_pct.loc[LAST_COMPLETE, 'Teens']:.0f}%.",
    f"**Genre tags overlap.** The most common combination is '{top_pair['genre_pair'].replace('  +  ', ' + ')}' ({top_pair['titles']:,} titles); titles carry {df['n_genres'].mean():.1f} genre tags on average.",
    f"**Each audience has its own genre fingerprint.** Kids: {top_genre_by_group['Kids']}. Teens: {top_genre_by_group['Teens']}. Adults: {top_genre_by_group['Adults']}.",
]
display(Markdown("\n\n".join(f"{i}. {t}" for i, t in enumerate(insights, 1))))

1. **Mature content dominates.** TV-MA is the most common rating (36% of titles), and 46% of the catalogue is Adult-rated.

2. **TV shows skew younger than movies.** 30% of TV titles target Kids/Older Kids versus 21% of movies, while Adult-rated content is 47% of movies and 43% of TV.

3. **The audience mix has shifted towards adults.** Adult-rated titles were 35% of 2010 releases and 54% of 2020 releases, while Teen-rated titles fell from 40% to 20%.

4. **Genre tags overlap.** The most common combination is 'Dramas + International Movies' (1,482 titles); titles carry 2.2 genre tags on average.

5. **Each audience has its own genre fingerprint.** Kids: Kids' TV, Children & Family Movies, Comedies. Teens: International Movies, Dramas, Comedies. Adults: International Movies, Dramas, International TV Shows.

### Business takeaways (draft – put them in your own words)
* **Adults are the core target.** Most of the library is TV-MA / TV-14 / R, so a family or kids viewer has far fewer choices – a possible opportunity for family content.
* **Kids content is concentrated in a few genres** (Kids' TV, Children & Family Movies); expanding into other genres could widen family appeal.
* **Because TV skews younger** than movies, series may be the better format for retaining households with children.

> ⚠️ **Limitations:** ratings describe *content maturity*, not who actually watches it; and genre tags are Netflix's own catalogue labels.

---
## ✅ Summary
Analysed rating categories and audience groups, identified top genres and genre pairs, compared ratings across Movies and TV Shows, built interactive Plotly visuals (bar, facet, sunburst, treemap, heatmap, area) and turned them into audience insights.
Interactive HTML versions of every chart are in `outputs/task5_rating_genre/`.